In [1]:
import arcpy
import os
import numpy as np
import pandas as pd
 

In [ ]:
#This script was used to run 1000 iterations of the random sampling method to create an expected dataset for the Sampling Score.

# Path to the STATE OUTLINE polygon
STATE_FC = r"C:\Users\email\OneDrive\Desktop\Auburn\Maps\Sampling_Bias_CH1\IteratingSamplingPlans.gdb\Alabama_Outline"
 
# Path to the watershed polygon feature class - used only to JOIN the random points to afterward and tally counts
WATERSHEDS_FC = r"C:\Users\email\OneDrive\Desktop\Auburn\Maps\Sampling_Bias_CH1\IteratingSamplingPlans.gdb\HUC10_Alabama"
WATERSHED_ID_FIELD = "huc10"
 
# Geodatabase to store the simulation outputs (point layers + summary)
OUTPUT_GDB = r"C:\Users\email\OneDrive\Desktop\Auburn\Maps\Sampling_Bias_CH1\IteratingSamplingPlans.gdb"
arcpy.env.overwriteOutput = True
 
# A subfolder/prefix for the saved random point layers from each iteration, so they're easy to find and distinguish from other outputs
POINTS_PREFIX = "randpts_iter"
 
# Number of simulation iterations 
N_SIMULATIONS = 1000
 
# Total number of samples observed across the study period of 2010-2025 for E. coli samples taken by ADEM
TOTAL_SAMPLES_ALL_YEARS = 23983
 
# Where to save the final per-watershed summary table
OUTPUT_CSV = r"C:\Users\email\OneDrive\Desktop\Auburn\Maps\Sampling_Bias_CH1\IteratingSamplingPlans.gdb\samples_simple_random_results.csv"
 

In [ ]:

def main():
    print(f"--- Running {N_SIMULATIONS} simple-random simulations across the state "
          f"boundary, {TOTAL_SAMPLES_ALL_YEARS} points per simulation, "
          f"then joining to watersheds ---")
 
    # Get the full list of watershed IDs so every watershed has a row even if a given simulation happens to place zero points in it
    all_ids = sorted({
        row[0] for row in arcpy.da.SearchCursor(WATERSHEDS_FC, [WATERSHED_ID_FIELD])
    })
    sim_counts = {wid: [] for wid in all_ids}
 
    for i in range(N_SIMULATIONS):
        pts_fc = os.path.join(OUTPUT_GDB, f"{POINTS_PREFIX}_{i}")
        joined_fc = os.path.join(OUTPUT_GDB, f"tmp_join_{i}")
 
        # 1. Generate simple random points across the STATE boundary
        arcpy.management.CreateSpatialSamplingLocations(
            in_study_area=STATE_FC,
            out_features=pts_fc,
            sampling_method="RANDOM",
            strata_id_field=None,
            strata_count_method="EQUAL",
            num_samples=TOTAL_SAMPLES_ALL_YEARS,
            geometry_type="POINT",
            min_distance=None,
        )
 
        # 2. Spatial join: one row per watershed, Join_Count = points inside it
        arcpy.analysis.SpatialJoin(
            target_features=WATERSHEDS_FC,
            join_features=pts_fc,
            out_feature_class=joined_fc,
            join_operation="JOIN_ONE_TO_ONE",
            join_type="KEEP_ALL",
            match_option="INTERSECT",
        )
 
        # 3. Read counts back per watershed
        with arcpy.da.SearchCursor(joined_fc, [WATERSHED_ID_FIELD, "Join_Count"]) as cur:
            for wid, count in cur:
                if wid in sim_counts:
                    sim_counts[wid].append(count if count is not None else 0)
 
        # 4. Delete both temp outputs - only the counts (read above) are kept to keep ArcGIS Pro running
        arcpy.management.Delete(pts_fc)
        arcpy.management.Delete(joined_fc)
 
        if (i + 1) % 50 == 0:
            print(f"    completed {i + 1}/{N_SIMULATIONS} iterations")
 
    # 5. Summarize mean/std per watershed across all simulations
    records = []
    for wid, counts in sim_counts.items():
        arr = np.array(counts, dtype=float)
        records.append({
            WATERSHED_ID_FIELD: wid,
            "sim_mean": arr.mean(),
            "sim_std": arr.std(ddof=0),
            "n_simulations": len(arr),
        })
 
    results_df = pd.DataFrame(records)


    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Summary table written to {OUTPUT_CSV}")
    print(f"All {N_SIMULATIONS} intermediate point layers were deleted after "
          f"each iteration's counts were recorded - only {OUTPUT_CSV} remains.")
    print("NOTE: join your observed sample counts (O_total) and add the "
          "Sampling Score calculation line above before final use.")
 
 
if __name__ == "__main__":
    main()

--- Running 1000 simple-random simulations across the state boundary, 23983 points per simulation, then joining to watersheds ---
    completed 50/1000 iterations
    completed 100/1000 iterations
